In [15]:
import os
import re
import easyocr
import pandas as pd
import google.generativeai as genai

from sentence_transformers import SentenceTransformer
from sentence_transformers import util

In [18]:
# EasyOCR
reader = easyocr.Reader(['en'])

# SBERT
sbert_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

# Gemini
genai.configure(
    api_key="AQ.Ab8RN6Jn7AHuF-WiYKDXbdJD5EwbAa8xRF8dZHXJs942xC-2cg"
)

gemini_model = genai.GenerativeModel(
    "gemini-2.5-flash"
)

print("All Models Loaded Successfully")

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

All Models Loaded Successfully


In [19]:
image_path = r"C:\Users\benan\OneDrive\Desktop\AI_Learning_Project\datastes\handwritten_ans\student_10_q1.jpg"

print("Selected File:")
print(image_path)

Selected File:
C:\Users\benan\OneDrive\Desktop\AI_Learning_Project\datastes\handwritten_ans\student_10_q1.jpg


In [20]:
result = reader.readtext(image_path)

extracted_text = " ".join(
    [item[1] for item in result]
)

print("Raw OCR Output:")
print(extracted_text)

C:\Users\benan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Raw OCR Output:
Student 10 Question Al helps machinesanalyze data and make predictions


In [22]:
cleaned_text = extracted_text

# Remove Student labels
cleaned_text = re.sub(
    r"Student\s*\d+",
    "",
    cleaned_text,
    flags=re.IGNORECASE
)

# Remove Question labels
cleaned_text = re.sub(
    r"Question\s*\d*",
    "",
    cleaned_text,
    flags=re.IGNORECASE
)

# OCR Corrections
ocr_corrections = {
    "Al": "AI",
    "machinesanalyze": "machines analyze",
    "machinelearning": "machine learning",
    "deeplearning": "deep learning"
}

for wrong, correct in ocr_corrections.items():
    cleaned_text = cleaned_text.replace(
        wrong,
        correct
    )

cleaned_text = cleaned_text.strip()

print("\nCleaned Answer:")
print(cleaned_text)


Cleaned Answer:
AI helps machines analyze data and make predictions


In [23]:
filename = os.path.basename(image_path)

# student_10_q1.jpg -> Q1
question_id = filename.split("_")[-1] \
                     .replace(".jpg", "") \
                     .upper()

print("Question ID:", question_id)

Question ID: Q1


In [24]:
answer_key = pd.read_csv(
    r"C:\Users\benan\OneDrive\Desktop\AI_Learning_Project\datastes\answer_key\answer_key.csv"
)

answer_key.head()

,Question_ID,Question,Reference_Answer,Max_Marks
0,Q1,What is Artificial Intelligence?,Artificial Intelligence is a branch of compute...,10
1,Q2,Explain Machine Learning.,Machine Learning is a subset of Artificial Int...,10
2,Q3,What is NLP?,Natural Language Processing is a field of Arti...,10
3,Q4,Explain Deep Learning.,Deep Learning is a subset of Machine Learning ...,10
4,Q5,What are Transformers?,Transformers are deep learning architectures t...,10


In [25]:
reference_answer = answer_key[
    answer_key["Question_ID"] == question_id
]["Reference_Answer"].values[0]

print("Reference Answer:")
print(reference_answer)

Reference Answer:
Artificial Intelligence is a branch of computer science that enables machines to simulate human intelligence, learn from data, reason, solve problems, and make decisions.


In [26]:
reference_embedding = sbert_model.encode(
    reference_answer,
    convert_to_tensor=True
)

student_embedding = sbert_model.encode(
    cleaned_text,
    convert_to_tensor=True
)

similarity = util.cos_sim(
    reference_embedding,
    student_embedding
)

similarity_score = float(similarity)

print("Similarity Score:")
print(round(similarity_score, 3))

Similarity Score:
0.622


In [27]:
marks = round(
    (similarity_score * 5) + 5,
    2
)

if marks > 10:
    marks = 10

print("Marks:", marks, "/10")

Marks: 8.11 /10


In [28]:
prompt = f"""
You are an academic evaluator.

Question ID:
{question_id}

Reference Answer:
{reference_answer}

Student Answer:
{cleaned_text}

Similarity Score:
{similarity_score}

Marks:
{marks}/10

Provide:

1. Strengths
2. Missing Concepts
3. Suggestions for Improvement

Keep the response concise and professional.
"""

response = gemini_model.generate_content(
    prompt
)

feedback = response.text

In [29]:
print("=" * 60)
print("AI BASED LEARNING COMPANION")
print("=" * 60)

print("\nQuestion ID:")
print(question_id)

print("\nStudent Answer:")
print(cleaned_text)

print("\nReference Answer:")
print(reference_answer)

print("\nSimilarity Score:")
print(round(similarity_score, 3))

print("\nMarks:")
print(marks, "/10")

print("\nFeedback:")
print(feedback)

print("\n" + "=" * 60)

AI BASED LEARNING COMPANION

Question ID:
Q1

Student Answer:
AI helps machines analyze data and make predictions

Reference Answer:
Artificial Intelligence is a branch of computer science that enables machines to simulate human intelligence, learn from data, reason, solve problems, and make decisions.

Similarity Score:
0.622

Marks:
8.11 /10

Feedback:
**1. Strengths**
The student answer correctly identifies two important functionalities of AI: data analysis and making predictions. It accurately attributes these capabilities to machines.

**2. Missing Concepts**
The answer lacks a fundamental definition of AI (e.g., as a branch of computer science). It also misses the core objective of AI, which is to simulate human intelligence. Additionally, it omits several key functionalities like learning from data (beyond just analysis), reasoning, solving problems, and broader decision-making processes.

**3. Suggestions for Improvement**
To improve, the student should:
*   Begin with a clear 